In [2]:
# --------------------------------------------------
# 03 - Feature Engineering for Credit Card Fraud
# --------------------------------------------------

import pandas as pd
import numpy as np
import yaml
import sys

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# --------------------------------------------------
# Load Config
# --------------------------------------------------

with open("../configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Config loaded")

# --------------------------------------------------
# Load Processed Dataset
# --------------------------------------------------

df = pd.read_csv(config["paths"]["processed_data"])

print("Dataset Shape:", df.shape)

# --------------------------------------------------
# Feature Scaling
# --------------------------------------------------

scale_columns = config["dataset"]["scale_columns"]

scaler = StandardScaler()

df[scale_columns] = scaler.fit_transform(df[scale_columns])

print("Scaled Columns:", scale_columns)

# --------------------------------------------------
# Feature / Target Split
# --------------------------------------------------

TARGET = config["dataset"]["target_column"]

X = df.drop(TARGET, axis=1)
y = df[TARGET]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

# --------------------------------------------------
# Train Test Split
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=config["random_state"],
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

# --------------------------------------------------
# Handle Class Imbalance using SMOTE
# --------------------------------------------------

smote = SMOTE(random_state=config["random_state"])

X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("\nBefore SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_res.value_counts())

# --------------------------------------------------
# Save Engineered Dataset
# --------------------------------------------------

engineered_path = "../data/processed/creditcard_engineered.csv"

X_train_res["Class"] = y_train_res

X_train_res.to_csv(engineered_path, index=False)

print("\nEngineered dataset saved to:", engineered_path)

Config loaded
Dataset Shape: (284807, 31)
Scaled Columns: ['Amount', 'Time']
Feature Shape: (284807, 30)
Target Shape: (284807,)
Train Shape: (227845, 30)
Test Shape: (56962, 30)

Before SMOTE:
Class
0    227451
1       394
Name: count, dtype: int64

After SMOTE:
Class
0    227451
1    227451
Name: count, dtype: int64

Engineered dataset saved to: ../data/processed/creditcard_engineered.csv
